# Substack Archive Processor

This notebook processes your Substack posts into:
- 📊 Searchable vector database
- 📝 AI-generated summaries
- 📖 Extracted glossary
- 🌐 Static website for GitHub Pages

## Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

# Setup paths
PROJECT_ROOT = Path('.').absolute().parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.config import Config
from pipeline.main import SubstackPipeline
from scripts.upload_handler import extract_zip

## Upload Your Posts

Upload a ZIP file containing your Substack posts (markdown, HTML, or JSON format).

In [ ]:
# Option 1: Upload via file dialog (works in Jupyter/Colab)
try:
    from google.colab import files
    uploaded = files.upload()
    zip_filename = list(uploaded.keys())[0]
    zip_path = Path(zip_filename)
    print(f"Uploaded: {zip_filename}")
except ImportError:
    # Not in Colab - use ipywidgets or specify path manually
    print("Not in Colab. Please set zip_path manually below.")
    zip_path = None

In [ ]:
# Option 2: Specify path manually
# zip_path = Path('/path/to/your/posts.zip')

# Option 3: Upload individual files to data/raw/ directory

## Extract ZIP File

In [ ]:
if zip_path and zip_path.exists():
    raw_data_dir = PROJECT_ROOT / 'data' / 'raw'
    extracted_files = extract_zip(zip_path, raw_data_dir)
    print(f"\nExtracted {len(extracted_files)} files")
else:
    print("Please upload a ZIP file or specify zip_path")

## Configure the Pipeline

In [ ]:
# Configuration
config = Config()

# Set your API key (or use .env file)
import os
os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'  # Replace with your key

print(f"Using LLM: {config.llm.provider} / {config.llm.model_name}")
print(f"Using Embeddings: {config.embedding.provider} / {config.embedding.model_name}")

## Run the Pipeline

You can run the full pipeline or individual steps.

In [ ]:
# Initialize pipeline
pipeline = SubstackPipeline(config)

In [ ]:
# Step 1: Load posts
posts = pipeline.load_posts()
print(f"Loaded {len(posts)} posts")

In [ ]:
# Preview loaded posts
for post in posts[:5]:
    print(f"- {post.title} ({post.word_count} words)")

In [ ]:
# Step 2: Create vector embeddings (for search)
pipeline.process_and_index()

In [ ]:
# Step 3: Generate summaries (uses LLM - may take a while)
summaries = pipeline.generate_summaries()

In [ ]:
# Step 4: Extract glossary terms
glossary = pipeline.extract_glossary()

In [ ]:
# Step 5: Generate collection summary
collection_summary = pipeline.generate_collection_summary()
print(collection_summary)

In [ ]:
# Step 6: Generate static site
# Set base_url for GitHub Pages (e.g., 'https://username.github.io/repo')
base_url = ''  # Leave empty for local viewing
pipeline.generate_site(base_url)

## Search Your Archive

In [ ]:
# Search the vector store
query = "machine learning"  # Change this to your search query
results = pipeline.search(query, k=5)

## Download Results

In [ ]:
# Create ZIP of generated site
import shutil

site_dir = PROJECT_ROOT / 'site'
output_zip = PROJECT_ROOT / 'generated_site.zip'

shutil.make_archive(
    str(output_zip.with_suffix('')),
    'zip',
    str(site_dir)
)

print(f"Site packaged to: {output_zip}")

In [ ]:
# Download in Colab
try:
    from google.colab import files
    files.download(str(output_zip))
except ImportError:
    print(f"Download manually from: {output_zip}")